# Practice Lab: Fine-Tuning on Vertex AI (Lesson 9.3)

Module 9 · 8 exercises · Vertex AI format + cost + hyperparameters + DPO + evaluation

Exercises 1, 2, 4, 5 run locally (pure Python).


# Lesson 9.3: Fine-Tuning on Vertex AI

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 4, 5 run **locally** (pure Python). Ex 3, 6, 7, 8 are design/architecture.


In [ ]:
import json
import numpy as np
np.random.seed(42)


---
## Exercise 1 (Easy): Vertex AI Data Format


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

def openai_to_gemini(ex):
    result = {"contents": []}
    for msg in ex["messages"]:
        if msg["role"] == "system":
            result["systemInstruction"] = {"parts": [{"text": msg["content"]}]}
        elif msg["role"] == "user":
            result["contents"].append({"role":"user","parts":[{"text":msg["content"]}]})
        elif msg["role"] == "assistant":
            result["contents"].append({"role":"model","parts":[{"text":msg["content"]}]})
    return result

data = [
    {"messages":[{"role":"system","content":"You are a Netsetos assistant."},
                 {"role":"user","content":"Refund policy?"},{"role":"assistant","content":"Full refund 7 days. 50% 8-30. None after 30."}]},
    {"messages":[{"role":"system","content":"You are a Netsetos assistant."},
                 {"role":"user","content":"Cost?"},{"role":"assistant","content":"14999 rupees with lifetime access."}]},
    {"messages":[{"role":"system","content":"You are a Netsetos assistant."},
                 {"role":"user","content":"Prerequisites?"},{"role":"assistant","content":"Basic Python and HS math."}]},
    {"messages":[{"role":"system","content":"You are a Netsetos assistant."},
                 {"role":"user","content":"EMI?"},{"role":"assistant","content":"Yes via Razorpay 2500/month."}]},
    {"messages":[{"role":"system","content":"You are a Netsetos assistant."},
                 {"role":"user","content":"Live classes?"},{"role":"assistant","content":"Daily 7 PM IST. Recordings in 2 hours."}]},
    # Multi-turn
    {"messages":[{"role":"system","content":"You are a Netsetos assistant."},
                 {"role":"user","content":"Courses?"},{"role":"assistant","content":"GenAI, DS, Cloud, Full-Stack."},
                 {"role":"user","content":"GenAI cost?"},{"role":"assistant","content":"14999 rupees lifetime."}]},
    {"messages":[{"role":"system","content":"You are a Netsetos assistant."},
                 {"role":"user","content":"Can I refund?"},{"role":"assistant","content":"When did you enroll?"},
                 {"role":"user","content":"5 days ago"},{"role":"assistant","content":"Full refund eligible."}]},
    {"messages":[{"role":"system","content":"You are a Netsetos assistant."},
                 {"role":"user","content":"Certificate?"},{"role":"assistant","content":"Yes, after all 14 modules."}]},
    {"messages":[{"role":"system","content":"You are a Netsetos assistant."},
                 {"role":"user","content":"Group discount?"},{"role":"assistant","content":"20% off for 3+ students."}]},
    {"messages":[{"role":"system","content":"You are a Netsetos assistant."},
                 {"role":"user","content":"Duration?"},{"role":"assistant","content":"146 hours, 14 modules."}]},
]

converted = [openai_to_gemini(ex) for ex in data]
with open("/tmp/vertex_train.jsonl","w") as f:
    for item in converted: f.write(json.dumps(item)+"\n")

mt = sum(1 for c in converted if len(c["contents"])>2)
ws = sum(1 for c in converted if "systemInstruction" in c)
errors = 0
for item in converted:
    for e in item["contents"]:
        if e["role"] not in ("user","model"): errors += 1
        if "parts" not in e: errors += 1

print(f"Vertex AI Format: {len(converted)} examples")
print(f"Single-turn: {len(converted)-mt} | Multi-turn: {mt} | With system: {ws}")
print(f"Validation: {errors} errors")
print(f"\nKey: 'assistant'->'model', 'messages'->'contents', string->parts[{{text}}]")


</details>


---
## Exercise 2 (Easy): Cost Estimator


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class VCost:
    TP = {"flash":4.0,"pro":12.0}  # train $/MTok
    IP = {"flash":(0.10,0.40),"pro":(1.25,5.00)}  # infer $/MTok in/out
    def train(self, n, tok=500, ep=3, m="flash"):
        return round(n*tok*ep*self.TP[m]/1e6, 2)
    def infer(self, qpm, months=6, m="flash"):
        pi,po = self.IP[m]
        pq = (500*pi+200*po)/1e6
        return round(pq*qpm*months, 2), round(pq*qpm, 2)
    def total(self, n, qpm, months=6, m="flash"):
        t = self.train(n, m=m)
        inf, mo = self.infer(qpm, months, m)
        return {"train":t,"infer":inf,"monthly":mo,"total":round(t+inf,2),"model":m}

vc = VCost()
print("Vertex AI Cost Estimator:")

print(f"\nTraining (3 epochs, 500 tok/ex):")
print(f"  {'Examples':<10} {'Flash':>8} {'Pro':>8}")
for n in [100,1000,10000]:
    print(f"  {n:<10} ${vc.train(n,m='flash'):>7.2f} ${vc.train(n,m='pro'):>7.2f}")

print(f"\nInference (50K/mo, 6 months):")
for m in ["flash","pro"]:
    inf, mo = vc.infer(50000, m=m)
    print(f"  {m.title()}: ${mo:.2f}/mo, ${inf:.2f} total. Premium: $0!")

print(f"\nTotal (1000 ex + 50K/mo, 6 months):")
for m in ["flash","pro"]:
    t = vc.total(1000, 50000, m=m)
    print(f"  {m.title()}: train=${t['train']} + infer=${t['infer']} = ${t['total']}")

print(f"\nVertex AI: $0 inference premium (vs OpenAI 50-100%)")
print(f"Free credits: $300 for new GCP accounts")


</details>


---
## Exercise 4 (Medium): Hyperparameter Comparison


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

def simulate(name, lr, epochs, adapter, n_ex):
    ofit = lr * adapter / max(n_ex/100, 1)
    tl = [max(2.0*np.exp(-lr*0.3*(e+1))+np.random.normal(0,0.02), 0.01) for e in range(epochs)]
    vl = []
    for e in range(epochs):
        v = 2.0*np.exp(-lr*0.2*(e+1))
        if e > epochs*0.4: v += ofit*(e-epochs*0.4)*0.1
        vl.append(max(v+np.random.normal(0,0.03), 0.01))
    bv = min(vl); be = vl.index(bv)+1
    return {"name":name,"tl":round(tl[-1],3),"vl":round(vl[-1],3),"bv":round(bv,3),"be":be,"ofit":vl[-1]>bv*1.1}

configs = [
    ("Default (Flash)",        1.0,  5, 4, 1000),
    ("Aggressive (Pro small)", 10.0, 20, 4, 500),
    ("Conservative",           0.5,  3, 2, 1000),
]

print("Hyperparameter Comparison:")
print(f"{'Config':<25} {'LR':>4} {'Ep':>4} {'Rank':>5} {'T.Loss':>7} {'V.Loss':>7} {'Best@':>6} {'Overfit'}")
print("-"*70)
for name,lr,ep,a,n in configs:
    r = simulate(name,lr,ep,a,n)
    print(f"  {name:<23} {lr:>4.1f} {ep:>4} {a:>5} {r['tl']:>7.3f} {r['vl']:>7.3f} {r['be']:>5} {'YES' if r['ofit'] else 'no'}")

print(f"\nPro + small data: LR=10, epochs=20 (Google's recommendation)")
print(f"Flash: defaults work. Adapter 4 = sweet spot.")
print(f"Val loss rising = checkpoint the best epoch.")


</details>


---
## Exercise 5 (Medium): DPO Preference Data


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

def score_pair(chosen, rejected):
    cw = set(chosen.lower().split()); rw = set(rejected.lower().split())
    overlap = len(cw&rw)/max(len(cw|rw),1)
    lr = len(chosen)/max(len(rejected),1)
    return round((1-overlap)*min(lr,2)/2, 2)

pairs = [
    ("Refund policy?","Full refund 7 days. 50% 8-30 days. None after 30. Contact support@netsetos.com.","We have a refund policy."),
    ("Course cost?","14999 rupees with lifetime access, Discord, GCP credits.","Check pricing page."),
    ("Prerequisites?","Basic Python and HS math. No ML needed. We teach from scratch.","Some background needed."),
    ("EMI?","Yes! Razorpay EMI starting 2500/month for 6 months.","Maybe. Ask support."),
    ("Live classes?","Daily 7 PM IST. Recordings within 2 hours.","Evening time."),
    ("Certificate?","Yes, completion certificate after 14 modules. Verifiable online.","Yes."),
    ("Tools learned?","Colab, Vertex AI, ChromaDB, LangChain, Docker, FastAPI.","Various AI tools."),
    ("Students?","5000+ enrolled. 85% completion. 4.8 stars.","Many students."),
    ("Mobile access?","Yes, all materials accessible on mobile via browser.","Should work."),
    ("Group discount?","20% off for 3+ students. Contact sales@netsetos.com.","Discounts sometimes."),
]

print("DPO Preference Data:")
total = 0
for prompt, chosen, rejected in pairs:
    s = score_pair(chosen, rejected)
    total += s
    print(f"  [{s:.2f}] {prompt[:25]}...")
    print(f"       C: {chosen[:35]}... | R: {rejected[:25]}...")

print(f"\nAvg contrast: {total/len(pairs):.2f}/1.0")

# Export Vertex DPO format
dpo = []
for prompt, chosen, rejected in pairs:
    dpo.append({"contents":[{"role":"user","parts":[{"text":prompt}]}],
                "chosen":{"role":"model","parts":[{"text":chosen}]},
                "rejected":{"role":"model","parts":[{"text":rejected}]}})

with open("/tmp/dpo_train.jsonl","w") as f:
    for item in dpo: f.write(json.dumps(item)+"\n")

print(f"Exported {len(dpo)} DPO pairs")
print(f"Workflow: Phase 1 SFT (format) -> Phase 2 DPO (quality)")


</details>


---
## Exercise 3 (Medium): Launch SFT Job (google.genai)
Design/architecture. See practice-lab-9_3.html.


In [ ]:
# Exercise 3: Launch SFT Job (google.genai)
pass


---
## Exercise 6 (Challenge): Evaluation Pipeline
Design/architecture. See practice-lab-9_3.html.


In [ ]:
# Exercise 6: Evaluation Pipeline
pass


---
## Exercise 7 (Challenge): Gemma vs Gemini
Design/architecture. See practice-lab-9_3.html.


In [ ]:
# Exercise 7: Gemma vs Gemini
pass


---
## Exercise 8 (Challenge): India DPDPA Compliance
Design/architecture. See practice-lab-9_3.html.


In [ ]:
# Exercise 8: India DPDPA Compliance
pass
